In [2]:
# ============================================================
# CELL 1 — Imports and setup
# ============================================================
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from IPython.display import display
import os
import sys
sys.path.append('..')

# Publication style
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

CITIES = ['manhattan', 'pittsburgh', 'philadelphia']
CITY_COLORS = {'manhattan': '#2196F3', 'pittsburgh': '#FF9800', 'philadelphia': '#4CAF50'}
LABEL_COLORS = {'Answerable': '#4CAF50', 'Ambiguous': '#FF9800', 'Contradictory': '#F44336'}
VARIANT_COLORS = {'mask_landmark': '#9C27B0', 'mask_directions': '#2196F3', 'mask_both': '#FF5722'}

REPORTS_DIR = '../reports/llm_audits'
DATA_DIR = '../data'

os.makedirs(f'{REPORTS_DIR}/figures', exist_ok=True)
print("✅ Setup complete")

✅ Setup complete


In [3]:
# ============================================================
# NOTEBOOK 1 — CELL 2: Load Manhattan labeled variants
# ============================================================
with open(f'{DATA_DIR}/manhattan/underspecified_variants_labeled.json') as f:
    manhattan_data = json.load(f)

print(f"✅ Loaded {len(manhattan_data)} Manhattan experiments")
print(f"Sample experiment keys: {list(manhattan_data[0].keys())}")
print(f"Sample variant keys: {list(manhattan_data[0]['variants'][0].keys())}")

✅ Loaded 5468 Manhattan experiments
Sample experiment keys: ['sample_id', 'city', 'original_text', 'extracted_category', 'extracted_direction', 'extracted_noun', 'start_node', 'gold_goal_node', 'gold_goal_lat', 'gold_goal_lon', 'variants']
Sample variant keys: ['type', 'text', 'removed_element', 'oracle_label', 'reachable_candidate_count', 'candidate_nodes']


In [4]:
# ============================================================
# NOTEBOOK 1 — CELL 3: Manhattan Oracle 2 label distribution
# ============================================================
all_variants = [
    {**{'exp_id': exp['sample_id'], 'city': exp['city'],
        'extracted_category': exp.get('extracted_category'),
        'extracted_noun': exp.get('extracted_noun')},
     **v}
    for exp in manhattan_data
    for v in exp['variants']
]
manhattan_variants_df = pd.DataFrame(all_variants)

print(f"Total variants: {len(manhattan_variants_df)}")
print(f"Experiments: {len(manhattan_data)}")
print(f"\nLabel distribution:")
label_counts = manhattan_variants_df['oracle_label'].value_counts()
label_pcts = manhattan_variants_df['oracle_label'].value_counts(normalize=True)
display(pd.DataFrame({'count': label_counts, 'pct': label_pcts.map('{:.1%}'.format)}))

Total variants: 16066
Experiments: 5468

Label distribution:


,count,pct
oracle_label,,
Contradictory,6283,39.1%
Ambiguous,6241,38.8%
Answerable,3542,22.0%


In [5]:
# ============================================================
# NOTEBOOK 1 — CELL 4: Manhattan breakdown by variant type
# ============================================================
print("=== Manhattan Oracle 2: Breakdown by Variant Type ===\n")
pivot = manhattan_variants_df.groupby(['type', 'oracle_label']).size().unstack(fill_value=0)
pivot['total'] = pivot.sum(axis=1)
for col in ['Answerable', 'Ambiguous', 'Contradictory']:
    if col in pivot.columns:
        pivot[f'{col}_%'] = (pivot[col] / pivot['total'] * 100).round(1)

display(pivot)

# Cross-city comparison
print("\n=== Cross-City Variant Type Comparison ===")
rows = []
for city in CITIES:
    path = f'{DATA_DIR}/{city}/underspecified_variants_labeled.json'
    if not os.path.exists(path):
        continue
    with open(path) as f:
        data = json.load(f)
    for vtype in ['mask_landmark', 'mask_directions', 'mask_both']:
        variants = [v for exp in data for v in exp['variants'] if v['type'] == vtype]
        labels = Counter(v['oracle_label'] for v in variants)
        total = sum(labels.values())
        if total == 0:
            continue
        rows.append({
            'city': city, 'variant_type': vtype, 'total': total,
            'Answerable_%': labels['Answerable']/total*100,
            'Ambiguous_%': labels['Ambiguous']/total*100,
            'Contradictory_%': labels['Contradictory']/total*100,
        })

cross_city_df = pd.DataFrame(rows)
display(cross_city_df.round(1))

=== Manhattan Oracle 2: Breakdown by Variant Type ===



oracle_label,Ambiguous,Answerable,Contradictory,total,Answerable_%,Ambiguous_%,Contradictory_%
type,,,,,,,
mask_both,597,1671,3031,5299,31.5,11.3,57.2
mask_directions,4615,116,568,5299,2.2,87.1,10.7
mask_landmark,1029,1755,2684,5468,32.1,18.8,49.1



=== Cross-City Variant Type Comparison ===


,city,variant_type,total,Answerable_%,Ambiguous_%,Contradictory_%
0,manhattan,mask_landmark,5468,32.1,18.8,49.1
1,manhattan,mask_directions,5299,2.2,87.1,10.7
2,manhattan,mask_both,5299,31.5,11.3,57.2
3,pittsburgh,mask_landmark,705,2.8,14.6,82.6
4,pittsburgh,mask_directions,668,2.8,91.8,5.4
5,pittsburgh,mask_both,668,1.6,13.5,84.9
6,philadelphia,mask_landmark,990,13.8,21.2,64.9
7,philadelphia,mask_directions,976,2.9,84.4,12.7
8,philadelphia,mask_both,976,9.5,9.0,81.5


In [6]:
# ============================================================
# NOTEBOOK 1 — CELL 5: Manhattan sample examples per label
# ============================================================
for label in ['Answerable', 'Ambiguous', 'Contradictory']:
    subset = manhattan_variants_df[manhattan_variants_df['oracle_label'] == label]
    print(f"\n{'='*60}")
    print(f"  {label.upper()} — {len(subset)} variants ({len(subset)/len(manhattan_variants_df):.1%})")
    print(f"{'='*60}")
    for _, row in subset.head(3).iterrows():
        print(f"  Type: {row['type']} | Category: {row.get('extracted_category')}")
        print(f"  Text: {row['text'][:100]}")
        print()


  ANSWERABLE — 3542 variants (22.0%)
  Type: mask_landmark | Category: CAFE
  Text: Head northeast to meet me at the [MASK] on East 49th Street. United Nations is on my south and a hot

  Type: mask_both | Category: CAFE
  Text: Head [DIR_MASK] to meet me at the [MASK] on [DIR_MASK] 49th Street. United Nations is on my [DIR_MAS

  Type: mask_landmark | Category: RESTAURANT
  Text: Meet me at the [MASK]. Go northwest until you reach the MacDougal-Sullivan Gardens Historic District


  AMBIGUOUS — 6241 variants (38.8%)
  Type: mask_landmark | Category: GARDEN
  Text: Can you meet me at the [MASK] on Liberty Street. It's located on the north side of the street a coup

  Type: mask_directions | Category: GARDEN
  Text: Can you meet me at the garden on Liberty Street. It's located on the [DIR_MASK] side of the street a

  Type: mask_directions | Category: CAFE
  Text: Head [DIR_MASK] to meet me at the cafe on [DIR_MASK] 49th Street. United Nations is on my [DIR_MASK]


  CONTRADICTORY — 62

In [7]:
# ============================================================
# NOTEBOOK 1 — CELL 6: Manhattan suspicious pattern checks
# ============================================================
print("=== Suspicious Pattern Checks ===\n")

# 1. Error labels
errors = manhattan_variants_df[manhattan_variants_df['oracle_label'] == 'error']
print(f"1. Error labels: {len(errors)} (should be 0)")

# 2. Missing oracle labels
missing_labels = manhattan_variants_df['oracle_label'].isna().sum()
print(f"2. Missing oracle labels: {missing_labels} (should be 0)")

# 3. Answerable with 0 reachable candidates
bad_answerable = manhattan_variants_df[
    (manhattan_variants_df['oracle_label'] == 'Answerable') &
    (manhattan_variants_df['reachable_candidate_count'] == 0)
]
print(f"3. Answerable with 0 reachable candidates: {len(bad_answerable)} (should be 0)")

# 4. Contradictory with >0 candidates
bad_contra = manhattan_variants_df[
    (manhattan_variants_df['oracle_label'] == 'Contradictory') &
    (manhattan_variants_df['reachable_candidate_count'] > 0)
]
print(f"4. Contradictory with >0 reachable candidates: {len(bad_contra)} (should be 0)")

# 5. Variant type distribution balance
print(f"\n5. Variant type counts:")
print(manhattan_variants_df['type'].value_counts())

# 6. mask_both should never exceed mask_directions count
mask_dir = (manhattan_variants_df['type'] == 'mask_directions').sum()
mask_both = (manhattan_variants_df['type'] == 'mask_both').sum()
print(f"\n6. mask_directions ({mask_dir}) >= mask_both ({mask_both}): {mask_dir >= mask_both} ✅")

print("\n✅ All checks passed" if (len(errors)==0 and missing_labels==0 
    and len(bad_answerable)==0 and len(bad_contra)==0) else "⚠️  Issues found")

=== Suspicious Pattern Checks ===

1. Error labels: 0 (should be 0)
2. Missing oracle labels: 0 (should be 0)
3. Answerable with 0 reachable candidates: 0 (should be 0)
4. Contradictory with >0 reachable candidates: 0 (should be 0)

5. Variant type counts:
type
mask_landmark      5468
mask_directions    5299
mask_both          5299
Name: count, dtype: int64

6. mask_directions (5299) >= mask_both (5299): True ✅

✅ All checks passed
